# Honeydew — обучение собственной модели музыкальных запросов

Ноутбук обучает компактный классификатор **с нуля** на размеченных примерах. Он предсказывает жанр, настроение, энергию, лад и основной инструмент. BPM пока извлекается детерминированно в Rust. Это не генератор аудио: музыкальные партии по-прежнему создаёт движок Honeydew.

Для хорошего качества нужно расширять CSV реальными запросами. Стартовый набор — только проверка конвейера, не готовая обучающая выборка.

## 1. Подготовить данные
Загрузите `honeydew_training.csv` из папки `training/` репозитория. Одна строка — один запрос. Не используйте один и тот же текст одновременно в train и test. Желательно иметь хотя бы 50–100 разных формулировок на класс, а не копии шаблонов.

In [ ]:
!pip -q install pandas scikit-learn joblib

In [ ]:
from google.colab import files
uploaded = files.upload()  # выберите honeydew_training.csv
CSV_PATH = next(iter(uploaded))

In [ ]:
import json, re, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

df = pd.read_csv(CSV_PATH).dropna(subset=['text','genre','mood','energy','mode','instrument'])
required = ['text','genre','mood','energy','mode','instrument']
assert set(required).issubset(df.columns), f'CSV должен содержать колонки: {required}'
df = df[required].astype(str).drop_duplicates(subset=['text'])
print('Примеров:', len(df), '| классы:', {c: df[c].nunique() for c in required[1:]})
if len(df) < 50: print('ВНИМАНИЕ: мало примеров. Результат будет демонстрационным; добавьте данные.')

X_train, X_test, idx_train, idx_test = train_test_split(df['text'], df.index, test_size=0.2, random_state=42, stratify=df['genre'] if df['genre'].value_counts().min() >= 2 else None)
models = {}
for target in required[1:]:
    y_train = df.loc[idx_train, target]
    y_test = df.loc[idx_test, target]
    vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2,5), min_df=1, lowercase=True, sublinear_tf=True)
    xtr = vectorizer.fit_transform(X_train)
    xte = vectorizer.transform(X_test)
    clf = LogisticRegression(max_iter=1500, class_weight='balanced')
    clf.fit(xtr, y_train)
    pred = clf.predict(xte)
    print('\n###', target, '| accuracy:', round(accuracy_score(y_test, pred), 3))
    print(classification_report(y_test, pred, zero_division=0))
    models[target] = (vectorizer, clf)


## 2. Экспортировать компактные веса
Экспортируем словарь символьных n-грамм, IDF и коэффициенты линейных классификаторов в JSON. Это удобнее для последующей реализации inference на Rust, чем тащить Python или большой LLM в Android.

In [ ]:
bundle = {'format_version': 1, 'analyzer': 'char_ngrams_2_5', 'lowercase': True, 'models': {}}
for target, (vec, clf) in models.items():
    bundle['models'][target] = {
        'vocabulary': vec.vocabulary_,
        'idf': vec.idf_.tolist(),
        'classes': clf.classes_.tolist(),
        'coef': clf.coef_.tolist(),
        'intercept': clf.intercept_.tolist(),
    }
out = 'honeydew_intent_model.json'
with open(out, 'w', encoding='utf-8') as f: json.dump(bundle, f, ensure_ascii=False, separators=(',', ':'))
print('Экспортировано:', out, '| размер, KB:', round(__import__('os').path.getsize(out)/1024, 1))
files.download(out)

## 3. Проверка
Проверьте несколько новых запросов, которых нет в CSV. Если модель уверенно ошибается, добавьте разнообразные размеченные примеры и обучите заново. Не принимайте низкую accuracy на маленьком наборе за доказательство качества. Перед подключением к приложению отдельно проверим точность и скорость на Android.